In [10]:
import pandas as pd
import re
import numpy as np

# ---------------------------
# Funktion: Messreihenname extrahieren
# ---------------------------
def extract_series_name(filename):
    """
    Extrahiert Messreihenname aus Dateiname, z.B.:
    '003-748.5sccm-tr200.asc' -> '003-748.5sccm-tr200'
    """
    base = filename.replace(".asc", "")
    return base  # ggf. hier Regex anpassen, falls noch mehr Text im Namen ist

# ---------------------------
# CSV einlesen
# ---------------------------
input_file = "alle_peaks_korrigiert.csv"
df = pd.read_csv(input_file)

# Prüfen, ob nötige Spalten vorhanden sind
required_cols = {"Datei", "m/z", "area"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"CSV muss folgende Spalten enthalten: {required_cols}")

# ---------------------------
# Messreihenname bestimmen
# ---------------------------
df["Messreihe"] = df["Datei"].apply(extract_series_name)

# ---------------------------
# Normierung pro Messreihe auf Argon (m/z ~ 40)
# ---------------------------
argon_mz = 40
tolerance = 0.2  # Akzeptiere z.B. 39.5–40.5 als Argon

def normalize_to_argon(group):
    # Argon-Peak innerhalb der Messreihe finden
    argon_rows = group[np.abs(group["m/z"] - argon_mz) <= tolerance]
    if argon_rows.empty:
        group["area_norm"] = np.nan
        print(f"⚠️ Kein Argon-Signal in Messreihe '{group['Messreihe'].iloc[0]}' gefunden.")
        return group
    
    # Wenn mehrere Argon-Peaks vorhanden → Mittelwert nehmen
    argon_area = argon_rows["area"].mean()
    
    # Normierung
    group["area_norm"] = group["area"] / argon_area
    return group

df_norm = df.groupby("Messreihe", group_keys=False).apply(normalize_to_argon)

# ---------------------------
# Ergebnis speichern
# ---------------------------
output_file = "alle_peaks_normiert.csv"
df_norm.to_csv(output_file, index=False)

print(f"✅ Normierte Peaks gespeichert in '{output_file}'")


✅ Normierte Peaks gespeichert in 'alle_peaks_normiert.csv'


C:\Users\adako\AppData\Local\Temp\ipykernel_4512\2041349568.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_norm = df.groupby("Messreihe", group_keys=False).apply(normalize_to_argon)
